<a id="top"></a>

# Demo: chain-of-thought made visible

```yaml
title: "Demo: chain-of-thought made visible"
keywords: chain-of-thought, cot, step by step, numbered scaffold, reasoning is tokens, teacher demo
estimated duration: 12
```

> **Lesson:** L06. Demo 1 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L06/demos_or_activities.md). Makes **live** Claude calls through the `potato_llm` seam — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running. A real model **varies**: if you run this yourself, expect the marble problem to fail zero-shot at least twice in five tries. Clear outputs before committing.
>
> Here's the line to hold onto: **reasoning is tokens on the page** — the model is the same across all three runs; only the prompt changed.

## Contents

- [1. Setup](#1-setup)
- [2. Zero-shot — answer only](#2-zero-shot--answer-only)
- [3. Free-form chain-of-thought](#3-free-form-chain-of-thought)
- [4. Numbered scaffold](#4-numbered-scaffold)
- [5. Takeaways](#5-takeaways)

## 1. Setup

You'll use a timing+token wrapper so the *cost* of reasoning stays visible to you, and one multi-step probability problem the model is unreliable on zero-shot.

> We call `client.achat(...)` — the **async** (awaitable) twin of `client.chat(...)`. Every `PotatoLLMClient` offers both; the course defaults to the async one, so you `await` model calls (a notebook cell can `await` at top level). *Why* async is the default is the K05 prework's "why async for agents."

In [ ]:
import time

from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message, PotatoLLMClient

# A problem current Claude models get wrong a meaningful fraction of the time zero-shot,
# but reliably right once they show the work. Pre-compute the answer for the slide.
PROBLEM = (
    "A bag contains 5 red, 7 blue, and 3 green marbles. I draw 3 without replacement. "
    "What is the probability all three are different colors? "
)
KNOWN_ANSWER = "105/455 = 21/91 ≈ 0.2308"

# Live client. The key is read through common.config (pydantic-settings over env/.env);
# constructing the client raises a clear error if ANTHROPIC_API_KEY is missing.
client: PotatoLLMClient = AnthropicClient()


async def run(label: str, prompt: str, *, max_tokens: int = 600) -> None:
    """Send a single user prompt, print the reply with token + latency cost."""
    start = time.perf_counter()
    reply = await client.achat([Message.user(prompt)], max_tokens=max_tokens, temperature=0.0)
    elapsed = time.perf_counter() - start
    print(f"=== {label} ===")
    print(reply.text)
    print(
        f"\n[out_tokens={reply.usage.output_tokens} "
        f"in_tokens={reply.usage.input_tokens} {elapsed:.1f}s]\n"
    )


print(f"setup ready (live client: {client.model})")
print("known answer:", KNOWN_ANSWER)

[↑ Back to top](#top)

## 2. Zero-shot — answer only

Ask for just the number. Check it yourself, ✓ or ✗, against the known answer.

In [ ]:
await run("zero-shot", PROBLEM + "Answer with a single number.", max_tokens=50)

[↑ Back to top](#top)

## 3. Free-form chain-of-thought

The **exact same problem**, with one appended instruction. Watch what happens: the model now shows its work — and usually lands the right number. Nothing about the model changed; the page just got longer.

In [ ]:
await run("free-form CoT", PROBLEM + "Let's think step by step.")

[↑ Back to top](#top)

## 4. Numbered scaffold

This time you're naming the steps yourself. More tokens to write, but the structure stays consistent run to run — the controllability dial from the lecture.

In [ ]:
scaffold = (
    PROBLEM
    + "Solve in this order: (1) count the total ways to draw 3 marbles, "
    + "(2) count the favorable outcomes where all three differ, (3) divide. "
    + "End with the final probability as a decimal."
)
await run("numbered scaffold", scaffold)

[↑ Back to top](#top)

## 5. Takeaways

- The model was **identical** across all three runs — only the prompt changed. The "thinking" is just more tokens conditioning the answer.
- Free-form CoT is cheapest to write; the numbered scaffold trades tokens for consistent structure.
- Watch the token counts: CoT typically costs **3–10× the output tokens** of zero-shot. Reasoning is not free — that is [Demo 4](L0609_lecture.ipynb)'s whole point.
- Scratchpad tags ([Demo 2](L0605_lecture.ipynb)) will make that buried final answer cleanly parseable.

[↑ Back to top](#top)